# INITIAL IMPORT

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import numpy as np
import gymnasium as gym
from src.config import Configuration


CONFIG = Configuration(
    n_training_episodes = 250_000,
    learning_rate = 0.7,
    n_eval_episodes = 1000,
    # gym_id = "Taxi-v3",
    gym_id = "MiniGrid-DoorKey-5x5-v0", 
    # rm_file= "rm_taxi.txt",
    rm_file= "rm_doorkey.txt",
    max_steps = 1000,
    gamma = 0.99,
    max_epsilon = 1.0,
    min_epsilon = 0.05,
    decay_rate = 0.0001,
    use_rm = False, 
    use_crm = False, 
)

# LOAD ENV

In [3]:
import minigrid
from src.envs import MiniGridDiscreteWrapper

env = gym.make(CONFIG.gym_id, render_mode="rgb_array")
env = MiniGridDiscreteWrapper(env)

state_space = env.observation_space.n
print("There are ", state_space, " possible states")
action_space = env.action_space.n
print("There are ", action_space, " possible actions")

There are  400  possible states
There are  7  possible actions


In [4]:
state, _ = env.reset()
# # Returns a tuple: (taxi_row, taxi_col, passenger_location, destination)
# taxi_row, taxi_col, passenger_location, destination = env.unwrapped.decode(state)

# print(f"Taxi row: {taxi_row}")
# print(f"Taxi column: {taxi_col}")
# print(f"Passenger location: {passenger_location}")
# print(f"Destination: {destination}")
state

np.int64(140)

# DEFINE Q-LEARNING

In [5]:
from src.models import RewardMachine
from src.envs import get_propositions_taxi, get_propositions_doorkey


In [6]:
from src.models import QTable

In [7]:
from src.models import train_qtable

In [8]:
from src.models import evaluate_agent

# Generate and train

In [9]:
rm = RewardMachine(CONFIG, CONFIG.rm_file)
rm.states

{0: {(1, ('k',), 1.0)},
 1: {(0, ('!k',), -5.0), (2, ('k', 'o'), 2.0)},
 2: {(3, ('g',), 20.0)}}

In [10]:
# events = get_propositions_taxi(env, state)
events = get_propositions_doorkey(env, state)

rm.step(events)

(0, -1, False)

In [11]:
qt = QTable(CONFIG, env, rm_file=CONFIG.rm_file if CONFIG.use_rm else None)
qt = train_qtable(CONFIG, qt, get_propositions_doorkey, env)

100%|██████████| 250000/250000 [37:29<00:00, 111.14it/s]


In [12]:
mean_reward, std_reward = evaluate_agent(CONFIG, qt, get_propositions_doorkey, env)
print(f"Mean_reward={mean_reward:.2f} +/- {std_reward:.2f}")

100%|██████████| 1000/1000 [00:12<00:00, 78.33it/s]

Mean_reward=0.00 +/- 0.00


In [13]:
qt.Qtable

array([[[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]]], shape=(1, 400, 7))

# Record video

In [14]:
from src.utils import record_video

record_video(CONFIG, qt, env, get_propositions_doorkey, video_name=f"{CONFIG.rm_file}_qtable_video.gif")

# Bench mark

In [15]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from src.config import Configuration
from src.models import QTable, evaluate_agent
from src.envs import get_propositions


def train_qtable_with_progress(config: Configuration, eval_every: int = 500):
    """Train and periodically evaluate to track learning evolution."""
    env = gym.make(config.gym_id, render_mode="rgb_array")
    qtable = QTable(config, env, rm_file="rm_taxi.txt" if config.use_rm else None)

    iterations = []
    eval_means = []
    eval_stds = []

    for episode in tqdm(range(config.n_training_episodes), desc=f"RM={config.use_rm}, CRM={config.use_crm}"):
        epsilon = config.min_epsilon + (config.max_epsilon - config.min_epsilon) * np.exp(-config.decay_rate * episode)
        state, _ = env.reset()
        qtable.reset_rm()

        terminated, truncated = False, False
        for _ in range(config.max_steps):
            action = qtable.epsilon_greedy_policy(state, epsilon, env)
            new_state, env_reward, terminated, truncated, _ = env.step(action)

            rm_done = qtable.update(
                state,
                action,
                env_reward,
                new_state,
                config.gamma,
                config.learning_rate,
                env,
                use_crm=config.use_crm,
            )

            if terminated or truncated or rm_done:
                break

            state = new_state

        if (episode + 1) % eval_every == 0 or episode == 0:
            mean_reward, std_reward = evaluate_agent(config, qtable, get_propositions)
            iterations.append(episode + 1)
            eval_means.append(mean_reward)
            eval_stds.append(std_reward)

    env.close()
    return qtable, np.array(iterations), np.array(eval_means), np.array(eval_stds)


def convergence_iteration(iterations, values, smooth_window=3, plateau_ratio=0.95, patience=3):
    """Estimate convergence as first iteration that reaches and maintains near-plateau performance."""
    if len(values) == 0:
        return None

    if len(values) < smooth_window:
        smoothed = values
        smooth_iters = iterations
    else:
        kernel = np.ones(smooth_window) / smooth_window
        smoothed = np.convolve(values, kernel, mode="valid")
        smooth_iters = iterations[smooth_window - 1:]

    best = np.max(smoothed)
    threshold = best - (1.0 - plateau_ratio) * max(1.0, abs(best))

    if len(smoothed) < patience:
        return int(smooth_iters[np.argmax(smoothed)])

    for i in range(len(smoothed) - patience + 1):
        if np.all(smoothed[i : i + patience] >= threshold):
            return int(smooth_iters[i])

    return int(smooth_iters[np.argmax(smoothed)])


# Shared benchmark settings
base_kwargs = dict(
    n_training_episodes=4_000,
    learning_rate=0.7,
    n_eval_episodes=200,
    gym_id="Taxi-v3",
    max_steps=1000,
    gamma=0.90,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
)

variants = [
    {"label": "No RM / No CRM", "use_rm": False, "use_crm": False},
    {"label": "RM / No CRM", "use_rm": True, "use_crm": False},
    {"label": "RM / CRM", "use_rm": True, "use_crm": True},
]

results = {}

for variant in variants:
    cfg = Configuration(
        **base_kwargs,
        use_rm=variant["use_rm"],
        use_crm=variant["use_crm"],
    )

    _, iters, means, stds = train_qtable_with_progress(cfg, eval_every=500)
    conv_it = convergence_iteration(iters, means, smooth_window=3, plateau_ratio=0.95, patience=3)

    results[variant["label"]] = {
        "iterations": iters,
        "mean_rewards": means,
        "std_rewards": stds,
        "convergence_iteration": conv_it,
    }

# Print convergence summary
for label, data in results.items():
    print(f"{label:15s} -> estimated convergence at episode {data['convergence_iteration']}")

# Plot evolution
plt.figure(figsize=(12, 6))
for label, data in results.items():
    x = data["iterations"]
    y = data["mean_rewards"]
    s = data["std_rewards"]

    plt.plot(x, y, marker="o", linewidth=2, label=label)
    plt.fill_between(x, y - s, y + s, alpha=0.12)

plt.title("Q-Learning Benchmark: Reward Evolution vs Training Episodes")
plt.xlabel("Training episodes")
plt.ylabel("Evaluation mean reward")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

ImportError: cannot import name 'get_propositions' from 'src.envs' (/home/turbotowerlnx/Documents/Master/TFM/TFM-Reinforcement-Learning-Reward-Machines/app/src/envs/__init__.py)

# Human playing

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
from IPython.display import clear_output

# 1. Initialize with rgb_array for visual frames
if 'env' not in globals():
    env = gym.make(CONFIG.gym_id, render_mode="rgb_array")
    state, _ = env.reset()

# --- HUMAN PLAY ---
# 0=South, 1=North, 2=East, 3=West, 4=Pickup, 5=Dropoff
my_action = 5
# ------------------

# Step the simulation
state, reward, terminated, truncated, _ = env.step(my_action)

# 2. Get the single frame
img = env.render()

# 3. Visualize
clear_output(wait=True)
plt.figure(figsize=(5,5))
plt.imshow(img)
plt.axis('off')
plt.show()

# Data output
taxi_row, taxi_col, p_loc, dest = env.unwrapped.decode(state)
print(f"Action: {my_action} | Reward: {reward}")
print(f"Row: {taxi_row}, Col: {taxi_col}, P_Loc: {p_loc}, Dest: {dest}")

if terminated or truncated:
    print("Goal reached or Resetting...")
    state, _ = env.reset()

NameError: name 'CONFIG' is not defined